In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [5]:
SYSTEM_PROMPT= """
You are an expert AI assistant in resolving user queries using chain of thought. 
You work on START, PLAN AND OUTPUT steps. 
You need to first PLAN what needs to be done. The PLAN can be multiple steps. 
Once you think enough PLAN has been done. finally you can give an OUTPUT.

Must follow instruction:
- strictly follow the Json output format
- only run one step at a time
-  The sequence of steps is START(where user give an input), PLAN (that can be multiple itmes) and finally
OUTPUT (which is going to be displayed to the user)

OUTPUT Json Format: 
{"step": "START"| "PLAN"| "OUTPUT", "content": "<string>"}

Example: 
START: Hey, Can you solve 2 + 3 * 5 / 10
PLAN: {"step": "PLAN", "content": "Seems user likes to solve a math problem"}
PLAN: {"step": "PLAN", "content": "Looks like its a combination of arithmetic problem which can be solved using BODMAS methon"}
PLAN: {"step": "PLAN", "content": "Yes, The BODMAS is correct way of solving it"}
PLAN: {"step": "PLAN", "content": "first we my divide by 5 by 10 which is 0.5"}
PLAN: {"step": "PLAN", "content": "next we my multiply by 0.5 by 3 which is 1.5"}
PLAN: {"step": "PLAN", "content": "then we my add by 1.5 to 2 which is 3.5"}
PLAN: {"step": "PLAN", "content": "Great, We finally have an answer which is 3.5"}
OUTPUT: {"step": "OUTPUT", "content": "The answer is 3.5"}
"""



import json
print("\n")

message_history = [{"role": "system", "content":SYSTEM_PROMPT}]
user_query = input("👉🏼")
message_history.append({"role": "user", "content":user_query})


response = gemini.chat.completions.create(
model = "gemini-3.1-flash-lite",
response_format={f"type": "json_object"}, 
messages=message_history
)

raw_result = response.choices[0].message.content
message_history.append({"role": "assistant", "content": raw_result})
parsed_result = json.loads(raw_result)

if isinstance(parsed_result, dict):
    if "steps" in parsed_result:
        parsed_result = parsed_result["steps"]
    else:
        parsed_result = [parsed_result]

for item in parsed_result:

    step = item.get("step", "")
    content = item.get("content", "")

    if step == "START":
        print(f"🔥 {content}")

    elif step == "PLAN":
        print(f"📝 {content}")

    elif step == "OUTPUT":
        print(f"\n🤖 {content}")








🔥 The user wants to solve the arithmetic expression 2 + 3 * 5 / 10.
📝 I will follow the Order of Operations (PEMDAS/BODMAS): Multiplication and Division first (from left to right), then Addition and Subtraction.
📝 Step 1: Multiply 3 by 5, which equals 15.
📝 Step 2: Divide the result (15) by 10, which equals 1.5.
📝 Step 3: Add 2 to the result (1.5), which equals 3.5.

🤖 The answer is 3.5.
